This file is where the real test comes in: will adding features tracking a games similarity to hti games result in better outcomes for the model?

In [34]:
import pandas as pd

df = pd.read_pickle('../data/cleaned_games.pkl')

df = df.sort_values('release_date').reset_index(drop=True)

I am using a pre-built neural network model to encode the test descriptions of the games in the data set to then create a numerical score to represent similarities the games decription has to games that have already been created that fall into the hit category. This, along with metadata is the crux of my model, and I would like to see if it actually results in a superior model.

In [ ]:
import re

def clean_description(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    
    # remove common noise phrases
    text = re.sub(r"collector'?s edition", "", text)
    text = re.sub(r"complete edition", "", text)
    text = re.sub(r"game of the year edition", "", text)
    text = re.sub(r"definitive edition", "", text)
    
    # remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)
    
    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


In [92]:

    df['clean_description'] = df['description'].apply(clean_description)

In [93]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(
    df['clean_description'].tolist(),
    batch_size=32,
    show_progress_bar=True
)

df['embedding'] = list(embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10125.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches:   0%|          | 17/3564 [00:02<09:18,  6.35it/s]


KeyboardInterrupt: 

In [94]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# pick a random game
idx = np.random.randint(len(df))

query_vec = df.iloc[idx]['embedding']
query_name = df.iloc[idx]['name']  # or title column

# compute similarity to all
sims = cosine_similarity([query_vec], list(df['embedding']))[0]

# get top 5 similar (excluding itself)
top_idx = np.argsort(sims)[-6:-1][::-1]

print("Query Game:", query_name)
print("\nMost similar games:")

for i in top_idx:
    print(df.iloc[i]['name'], "| sim:", round(sims[i], 3))

Query Game: Esper - Make You Live Again

Most similar games:
Survival Diary | sim: 0.473
The Merchant Memoirs | sim: 0.473
Tormenta do Tempo | sim: 0.469
Slash Abyss | sim: 0.455
Complex Loop | sim: 0.454


Now I will take the embedded vectors to return a game's "similarity score" calculated with a cosine similarity function on the vectors compared with the vectors of all games released before that fall into "hit" and store the maximum similarity in a new column. I will also store the time since the hit game as a second metric to match with the similarity score.

In [95]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity_scores = []
time_since_similar = []

past_embeddings = []
past_hits = []
past_dates = []

for i, row in df.iterrows():
    current_vec = row['embedding']
    current_date = row['release_date']
    
    # indices of past successful games
    hit_indices = [j for j in range(len(past_hits)) if past_hits[j] == 1]
    
    if len(hit_indices) == 0:
        sim = 0
        time_gap = 0
    else:
        hit_vectors = [past_embeddings[j] for j in hit_indices]
        
        sims = cosine_similarity([current_vec], hit_vectors)[0]

        # --- Top-K similarity ---
        k = min(5, len(sims))
        top_k_idx = np.argsort(sims)[-k:]

        top_sims = sims[top_k_idx]

        # map back to original indices
        top_hit_indices = [hit_indices[idx] for idx in top_k_idx]

        top_time_gaps = np.array([
            (current_date - past_dates[j]).days
            for j in top_hit_indices
        ])

        weights = 1 / (1 + top_time_gaps)

        sim = np.sum(top_sims * weights) / np.sum(weights)

        # --- Best match for time feature ---
        best_local_idx = np.argmax(sims)
        best_global_idx = hit_indices[best_local_idx]

        past_date = past_dates[best_global_idx]
        time_gap = (current_date - past_date).days
    
    similarity_scores.append(sim)
    time_since_similar.append(time_gap)
    
    # update history
    past_embeddings.append(current_vec)
    past_hits.append(row['hit'])
    past_dates.append(current_date)

# final features
df['sim_to_past_hits'] = similarity_scores
df['log_days_since_similar_hit'] = np.log1p(time_since_similar)

In [96]:
df['sim_dev_interaction'] = df['sim_to_past_hits'] * df['dev_success']
df['sim_time_interaction'] = df['sim_to_past_hits'] * (1 / (1 + df['log_days_since_similar_hit']))

In [97]:
import pickle
import pandas as pd

with open('../data/genre_columns.pkl', 'rb') as f:
    genre_features = pickle.load(f)

features_clean = [
    'price',
    'year',
    'dev_success',
    'dev_had_success',
    'log_dev_experience',
    'num_devs',
    'sim_to_past_hits',
    'log_days_since_similar_hit',
] + list(genre_features)


X = df[features_clean_3]
y = df['hit']

split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]



In [98]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=3000,
    class_weight='balanced'
)

model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [99]:
threshold = 0.5

import numpy as np

y_probs = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_probs > threshold).astype(int)


In [101]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9421238782847622
F1 Score: 0.2161520190023753
              precision    recall  f1-score   support

           0       0.97      0.97      0.97     33069
           1       0.20      0.24      0.22      1142

    accuracy                           0.94     34211
   macro avg       0.59      0.60      0.59     34211
weighted avg       0.95      0.94      0.94     34211

[[31958  1111]
 [  869   273]]


In [102]:
import pandas as pd

coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
})

coef_df = coef_df.sort_values(by='coefficient', ascending=False)

print(coef_df.head(15))

                  feature  coefficient
2             dev_success     0.433668
22           Free To Play     0.185909
6        sim_to_past_hits     0.174310
3         dev_had_success     0.172206
30                    RPG     0.162122
8     sim_dev_interaction     0.115148
34             Simulation     0.107548
37               Strategy     0.102667
13              Adventure     0.100109
12                 Action     0.068797
26  Massively Multiplayer     0.057678
0                   price     0.047256
42         Web Publishing     0.042373
5                num_devs     0.027668
27                  Movie     0.023909
